# Aggregating Data

You will undoubtedly want to aggregate your data in some fashion. You may be given the data in separate files or in various tables in a database. Some of the common aggregation tasks include:

- Querying
- Merging
- Binning
- Applying functions
- Summarizing
- Creating pivot tables and crosstabs

We will explore these tasks in this section.

## Querying Data

You will encounter two basic ways to query data when using `pandas`. One approach is to use a Boolean mask to get the results you want. The other approach is to use the `.query()` function. This function makes it easy to write complicated filters instead of using a Boolean mask. We will examine both approaches using the weather data that we cleaned from the last section. Recall that the results of our cleaning efforts were saved in the file `./data/cleaned_data.csv`.

In [ ]:
# Import pandas
import pandas as pd

In [ ]:
# Read in the cleaned_data from our last section
wburg_weather = pd.read_csv('./data/cleaned_data.csv')

# Look at its info()
wburg_weather.info()

We want to find all the hourly data where the wind direction was variable, the dry bulb temperature in Fahrenheit was between 75 and 80 degrees (inclusively), and the station pressure was less than 29.5. To use a Boolean mask, we use the bitwise **and** operator, which is the ampersand character, `&`. We also need to put each condition inside of parentheses. 

In [ ]:
# Using Boolean mask
wburg_weather[(wburg_weather.HOURLYWindDirection == 'Variable Direction')
             & (wburg_weather.HOURLYDRYBULBTEMPF >= 75) 
             & (wburg_weather.HOURLYDRYBULBTEMPF <= 80)
             & (wburg_weather.HOURLYStationPressure < 29.5)]

When we use the `.query()` method, we can use both logical operators (`and`, `or`, `not`) and bitwise operators (`&`, `|`, `~`). In our case, we can use the logical operator `and`, which makes reading our statement easier. You should be able to verify that the results of both approaches are the same.

In [ ]:
# Use the .query() method instead
wburg_weather.query('HOURLYWindDirection == "Variable Direction" '
                   'and HOURLYDRYBULBTEMPF >= 75 and HOURLYDRYBULBTEMPF <= 80 '
                   'and HOURLYStationPressure < 29.5')

<hr style="border:1px solid gray">

<font color='red' size = '5'> Student Exercise </font>

Continuing with our weather data, complete the following tasks:

1. Using a Boolean mask, find all the rows that have hourly sky conditions of `CLR:00`, a dew point temperature greater than 75, and the relative humidity of either 94 or 100. Save your results in the variable `my_mask`.
2. How many rows does `my_mask` have?
3. Using the `.query()` method, perform the same task as above. Save the results in the variable `my_query`.
4. How many rows does `my_query` have?
5. Verify that the two `DataFrame`s contain the same data.

In [ ]:
# 1. Using a Boolean mask, find all the rows that have hourly sky conditions of `CLR:00`,
# a dew point temperature greater than 75, and the relative humidity of either 94 or 100.
# Save your results in the variable `my_mask`.


In [ ]:
# 2. How many rows does `my_mask` have?


In [ ]:
# 3. Using the `.query()` method, perform the same task as above. 
# Save the results in the variable `my_query`.


In [ ]:
# 4. How many rows does `my_query` have?


In [ ]:
# 5. Verify that the two `DataFrame`s contain the same data.


<hr style="border:1px solid gray">

## Merging `DataFrame`s

We have seen the ability to combine `DataFrame`s using `pd.concat()` and `append()`. Merging `DataFrame`s effectively deals with how we line them up by row. If you know anything about databases, you will recognize the merge concept as a **join**. There are four basic types of joins: inner, left, right, and full (outer). These are depicted in the picture below.

<img src="images/joins.png" alt="Four basic join types" width=50%>

The function `merge()` is how we merge or join `DataFrame`s. The default join type is an inner join, which is often the one you want. It will return only the rows from the two different datasets that have the same value in the column you specify. We are going to explore a relatively "large" dataset contained in the file `./data/store_sales.xlsx`. There are six total worksheets and we are going to read them all in at once. Then we will explore the concept of merging/joining those datasets to help answer several questions.

In [ ]:
# Now let's trying reading in all the sheets
# NOTE: This may take a minute or so ... be patient
store_sales = pd.read_excel('./data/store_sales.xlsx', sheet_name=None)

In [ ]:
# How many sheets total?
print(f'There are {len(store_sales)} worksheets')

In [ ]:
# What are their names?
print(store_sales.keys())

We are interested in the datasets `customers` and `transactions`. Let's first see how big each is. We'll do this two ways, first by calling `shape` and then by using `.info()`.

In [ ]:
# Rows and columns of customers
store_sales['customers'].shape

In [ ]:
# Rows and columns of transactions
store_sales['transactions'].shape

In [ ]:
# Get info for customers
store_sales['customers'].info()

In [ ]:
# Get info for transactions
store_sales['transactions'].info()

It looks like both datasets have the column `CustomerID`. I would even hazard a guess that the `CustomerID` in the customers dataset is the unique identifier. The `CustomerID` in the transactions dataset hopefully corresponds to one of the values in the customers dataset. We notice in both cases that the column `CustomerID` is an object. Recall that we can use `.describe()` on a `Series` of data type object to see a few interesting pieces of information. Let's try it for `CustomerID` for both datasets.

In [ ]:
# Call .describe() on customers.CustomerID
store_sales['customers']['CustomerID'].describe()

In [ ]:
# Call .describe() on transactions.CustomerID
store_sales['transactions']['CustomerID'].describe()

Interestingly, our customer dataset has 44,531 unique customers, but the transactions dataset only has 44,501 unique customers. What does this mean? It appears that there were 30 customers listed in the customers dataset that did not have any corresponding records in the transactions dataset.

### Inner Join

If we merge the two datasets with an inner join, what will be the result? That is, how many rows and columns will there be in the merged dataset? An inner join will keep only the records from each dataset where the specified column is the same in each dataset. Let's merge on `CustomerID` with an inner join. There are various ways to accomplish this task. We will first try it with the high-level `pandas` function `pd.merge()`. Then we'll try it by calling merge on one of the `DataFrame`s itself to see if there is any difference in the results.

In [ ]:
# Create inner join with high-level pd.merge
inner_join = pd.merge(store_sales['customers'],
                      store_sales['transactions'],
                      left_on='CustomerID',
                      right_on='CustomerID')

In [ ]:
# See the number of rows, columns
inner_join.shape

In [ ]:
# Take a random sample of 5 rows
inner_join.sample(5, random_state=42)

In [ ]:
# We can shorten to the following because CustomerID
# has the same name in both datasets
inner_join2 = pd.merge(store_sales['customers'],
                      store_sales['transactions'],
                      on='CustomerID')

In [ ]:
# See the number of rows, columns
inner_join2.shape

In [ ]:
# Take a random sample of 5 rows
inner_join2.sample(5, random_state=42)

You may have noticed that the number of rows from our inner join was 263,887. This is curious. We had 263,890 rows in the transactions dataset. Somehow, we lost three transactions. The explanation is that the those three transactions must have a `CustomerID` not found in the customers dataset. One way to see if this is the case is to call `.describe()` on `inner_join.CustomerID` and see how many unique customers we have.

In [ ]:
inner_join.CustomerID.describe()

We see that there are 44,500 unique customers in the `inner_join` dataset. If you look above, we had 44,501 unique customers in the tranactions dataset. This confirms that there must be a customer in the transactions dataset that is not present in the customers dataset. (Thought Exercise: Can you tell how many transactions this mystery customer had?)

Let's try another way to construct an inner join.

In [ ]:
# Call merge from the DataFrame instead of high-level pd
inner_join3 = store_sales['customers'].merge(store_sales['transactions'],
                                            on='CustomerID')

# See how many rows, columns
inner_join3.shape

### Left Join

A left join will keep all the rows of the left dataset, regardless if the column you join on is present in the right dataset. Let's use customers as our left dataset and merge on `CustomerID` as a left join to see what result we get.

In [ ]:
# Start with the DataFrame you want all the rows of
# then merge with the other DataFrame on a specific
# column being sure to use how='left'
left_join = store_sales['customers'].merge(store_sales['transactions'],
                                           on='CustomerID', how='left')

In [ ]:
# How many rows, columns?
left_join.shape

In [ ]:
# See how many customers exist in left_join
left_join.CustomerID.describe()

Our variable `left_join` contains all 44,531 the customers. If a customer does not have any transactions, those rows will be empty. Let's verify this.

In [ ]:
# See which columns have missing data
left_join.isnull().any()

How many customers do not have transactions associated with them but exist in the customers dataset? We can filter the dataset based on the fact that the `TXN-ID` column has missing data. (Note: You could use any of the four columns that indicated there was missing data.) It should not surprise you that the columns containing missing data are the ones that originally came from the transactions dataset - the dataset on the right of our left join.

In [ ]:
# how many customers in the customers dataset do not have transactions?
left_join[left_join['TXN-ID'].isnull()].shape

In [ ]:
# Get those 31 customers
left_join[left_join['TXN-ID'].isnull()]

### Right Join

A right join says keep all the records from the right dataset regardless if there is a corresponding row in the left dataset. We know that the transactions dataset contains 263,890 rows of data. If we use that as our right dataset, then we would expect that many rows in the resulting dataset. Let's try it.

In [ ]:
# Verify how many transactions we have
store_sales['transactions'].shape

In [ ]:
# Create a right join. 
# Here we are starting the same way as the left join,
# but the difference is we use how='right'
right_join = store_sales['customers'].merge(store_sales['transactions'],
                                            on='CustomerID', how='right')

In [ ]:
# How many rows, columns?
right_join.shape

In [ ]:
# Let's see how many unique customer ids
right_join.CustomerID.describe()

We see that we have 44,501 unique customers in our dataset `right_join`. This is one more customer than was in our inner join results. This means that we will have missing customer information for some of our transactions. It will be missing for however many transactions are associated with that mystery customer. The columns with missing data will all be from the dataset that was on the left (the customer dataset). Let's see which columns have missing data.

In [ ]:
# The columns with missing data should be the ones from customers
right_join.isnull().any()

In [ ]:
# How many transactions have a customer 
# that is missing from the customer dataset?
right_join[right_join['FirstName'].isnull()].shape

In [ ]:
# What were those transactions?
right_join[right_join['FirstName'].isnull()]

### Outer Join

An outer join gives you back the rows in both datasets. This means that you will potentially have rows where the customer information is missing or rows where the transaction data is missing. To create an outer join, use the argument `how='outer'` during the merge.

In [ ]:
# Create an outer join with how='outer'
outer_join = pd.merge(store_sales['customers'],
                      store_sales['transactions'],
                      on='CustomerID', how='outer')

In [ ]:
# How many rows, columns?
outer_join.shape

In [ ]:
# Which columns contain missing data?
outer_join.isnull().any()

In [ ]:
# Could also see this with .info()
outer_join.info()

So, we see that the customer information (except `CustomerID`) is missing for three rows. We also see that there are 31 rows missing the transaction information. 

### A Bit More Digging

One of the issues with merges and joins is that they can be resource-intensive. You may go through a lot of trial and error as you try to extract the desired results from a merge. With these operations being somewhat "expensive", you may waste a lot of time (and computing power) during your trial and error approach. To help mitigate wasting time, you should try to determine what the results of your proposed merge will look like. That is, how many rows and columns will be in the result. 

If you remember back to an earlier module, we discussed the `set` data structure. At that point, you were encouraged to explore the mathematical operations of sets. Here, we will use a few of those to help us understand potential joins and to more easily find how many missing objects we have.

First, we are going to set the index to both the customers and transactions datasets. In both instances, we will use the `CustomerID` as the index. Recall that an index is by definition a set. 

In [ ]:
# Create a new DataFrame named 'cust' and 
# set its index to be CustomerID
cust = store_sales['customers'].set_index('CustomerID')

# Show info
cust.info()

In [ ]:
# Look at
cust

In [ ]:
# Create a DataFrame named 'txn' and
# set its index to CustomerID
txn = store_sales['transactions'].set_index('CustomerID')

# See info
txn.info()

In [ ]:
# Look at it
txn

If we want to see how many unique customers would result in an inner join between customers and transactions based on the `CustomerID` column, we can use the **intersection** set operation. This concept should make sense if you look back at our Venn diagrams showing the different joins. 

In [ ]:
# Find the intersection of cust.index and txn.index
cust.index.intersection(txn.index)

The length shows that there will be 44,500 customers in the resulting inner join. However, it does not tell us which customers will be missing. Notice there were 31 missing customers from our inner join and this intersection. To find those customers, we can use the set operation **difference**.

In [ ]:
# Find the customers that do not have transactions
cust.index.difference(txn.index)

In [ ]:
# What about the other direction?
# Which customers have transactions but were not in the customers dataset?
txn.index.difference(cust.index)

In [ ]:
# Get all the transactions for 'cust-unknown'
txn.loc['cust-unknown']

In [ ]:
# Can we get all the customers who don't have transactions?
no_txns = cust.loc[cust.index.difference(txn.index)]
no_txns.shape

In [ ]:
no_txns

<hr style="border:1px solid gray">

## Binning

There are times when you want to take a variable that is continuous (or discrete with a lot of valid values) and create a new variable that creates "buckets" (bins or categories). For example, you might be more interested in analyzing the daily high temperature as one of several categories instead of a numerical quantity. Perhaps you think any temperature above 90&deg; F is "hot". You could create five different "buckets" to classify temperature as "cold", "cool", "just right", "warm", or "hot". How you decided to discretize a continuous variable is highly dependent on how you plan to use the new variable in your analysis. Here we will explore a couple of ways to bin our data.

We will be using the daily stock trading data for the company Apple for the year 2022. The data is in the file `./data/aapl_2022.csv`. Let's read it in trying to take care of the commas in the numerical columns and being sure to create a real date for the `Date` column.

In [ ]:
# Read in the Apple trading data
apple = pd.read_csv('./data/aapl_2022.csv', thousands=',', parse_dates=['Date'])
apple.info()

In [ ]:
# Change all column names to lower case
apple.columns = apple.columns.str.lower()
apple.info()

In [ ]:
# Change the index to the date
apple.set_index('date', inplace=True)
apple.info()

We are interested in investigating the volume of trading for Apple's stock in the year 2022. In particular, we want to calculate the absolute value of the Z-score for volume. Recall from statistics that the Z-score tells you how far away a particular observation is from the mean of all the observations for that variable. It tells you how many standard deviations away a data point is from the average. If your data follows a Normal distribution, then you would expect approximately 99.7% of the observations to fall between -3 and +3 standard deviations from the mean. Let's see if we have any days of trading volume that are outside of three standard deviations, in absolute terms, for Apple.

In [ ]:
# Find the absolute value of Z-score for volume
apple = apple.assign(abs_z_score_vol 
                     = lambda x: x.volume.sub(x.volume.mean()).div(x.volume.std()).abs())

In [ ]:
# Which days had volumes outside of 3 standard deviations?
apple.query('abs_z_score_vol > 3')

In [ ]:
# Examine trading days around the May 12th date
apple.loc['2022-05-10':'2022-05-16']

You may also be interested in the percentage change of a variable. You can use the `.pct_change()` function on either the `DataFrame` or a `Series`. This function will calculate the percentage change from the immediately preceding row by default. In our case with time series data, this is the action we want. In addition to finding the percentage change, we are also going to create a column that ranks the percentage change. The ranking column allows us to sort the data more easily.

In [ ]:
# See the percent change for volume and rank it
# This returns a new DataFrame of the 10 largest
# volume changes in percentage change from the previous trading day
apple.assign(vol_pct_change = apple.volume.pct_change(),
            pct_change_rank=lambda x: x.vol_pct_change.abs().rank(ascending=False)
            ).nsmallest(10, 'pct_change_rank')

Now we want to try to create bins or buckets for the daily trading volume. If you want the ranges defining the bins to be equidistant, then you can use the `pd.cut()` function. Let's try breaking the column `volume` into three equidistant bins. We can use the argument `bins=3` and `pandas` will find the break points for the bins for us. You can optionally label each bin, which we will do as "low", "medium", and "high". If you want both the binned data as well as the ranges, you use the argument `retbins=True`. The first element of the returned tuple will be the binned data and the second element will contain the defining break points for the bins. 

In [ ]:
# Unpack the returned tuple
# First element is the data that has been binned
# Second element is the break points of the bins
vol_binned, bin_ranges = pd.cut(apple.volume, bins=3,
                                labels=['low','medium','high'],
                                retbins=True)

In [ ]:
# See the break points for the bins
bin_ranges

In [ ]:
# See the range for each of 3 bins
for i in range(len(bin_ranges)-1):
    print(f'size of bin is: {bin_ranges[i+1] - bin_ranges[i]}')

In [ ]:
# See how many rows are in each bucket
vol_binned.value_counts()

In [ ]:
# Look at the 13 high volume days
apple[vol_binned=='high'].sort_values('volume', ascending=False)

Another way to create buckets is to try to have the same number of observations in each bucket. Doing so necessitates that the breakpoints for the bins **not** be equidistant. We can accomplish breaking a continuous variable into a discretized variable with each category containing (roughly) the same number of observations with the `pd.qcut()` function. The `q` stands for **quantile**. Using the argument `q=4` will give you back the quartiles; using `q=10` will return deciles. Let's break the `volume` variable into quartiles.

In [ ]:
# Equal number of observations in each bin
# Unpack the returned tuple
# First element is the data that has been binned
# Second element is the break points of the bins
vol_qbinned, qbin_ranges = pd.qcut(apple.volume, q=4,
                                   labels=['q1','q2','q3','q4'],
                                   retbins=True)

In [ ]:
# How many in each bin? Should be approximately same
vol_qbinned.value_counts()

In [ ]:
# See the break points for qcut
qbin_ranges

In [ ]:
# See the range for each of 4 bins (quartiles)
for i in range(len(qbin_ranges)-1):
    print(f'size of bin is: {qbin_ranges[i+1] - qbin_ranges[i]}')


In [ ]:
# Call .describe on the volume
# The 5 numbers for the qcut should be readily apparent
apple.volume.describe()

<hr style="border:1px solid gray">

## Applying Functions

We saw the use of the `apply()` function in the last section when we changed the wind direction values from `VRB` to `Variable Direction` in our weather dataset. The `apply()` function runs vectorized operations on entire columns (like we did) or rows at once. You can apply any function you want as long as the operations within the function are valid for your particular data. The `apply()` function is very efficient in terms of time complexity. It is a much better option than the alternative functions of `iteritems()`, `itertuples()`, or `iterrows()`. Pandas, which is built on top of NumPy, is designed for vectorized operations. Because of this, you should attempt to use `apply()` rather than looping through the `DataFrame` manually.

Even though we have an efficient way to bin a continuous variable with `pd.cut()` or `pd.qcut()`, let's try writing a custom function that will perform discretization. Then we will `.apply()` that function to a column (`Series`). 

In [ ]:
# Look at apple.info()
apple.info()

In [ ]:
# Call .describe() on apple
apple.describe()

In [ ]:
# Function that will check the value of the column
# 'close' and split it into one of three buckets
def create_buckets(row):
    # Each row is coming in with all columns
    if row['close'] <= 130:
        return 'low'
    elif row['close'] <= 160:
        return 'middle'
    else:
        return 'high'

In [ ]:
# Call .apply on the DataFrame being  
# sure to set the axis='columns'
apple.apply(create_buckets, axis='columns')

In [ ]:
# This function assumes you call .apply
# on a Series object
def create_buckets2(value):
    if value <= 130:
        return 'low'
    elif value <= 160:
        return 'middle'
    else:
        return 'high'

In [ ]:
# Call apply on the close column
# and apply create_buckets2
apple.close.apply(create_buckets2)

In [ ]:
# We can use create_buckets2 with any
# numerical column
apple.open.apply(create_buckets2)

In [ ]:
apple = apple.assign(close_text = apple.close.apply(create_buckets2))
apple

In [ ]:
# Add another column named 'close_text2' by using pd.cut
# This should result in same values as our custom function
apple = apple.assign(close_text2 = pd.cut(apple.close, 
                                          bins=[apple.close.min()-1,130,160,apple.close.max()+1],
                                          labels=['low','middle','high']))
apple

In [ ]:
# Verify both approaches did the same binning
(apple.close_text != apple.close_text2).sum()

<hr style="border:1px solid gray">

## Summarizing `DataFrame`s

There are times when we want to summarize various columns of a `DataFrame`. Sometimes you want the same summary statistics for all the numerical columns. Other times, you might want different summary statistics for different columns. If you want the same summary statistic, e.g., the average of each column, you can use the `.apply()` function discussed above. Remember that each column data type must be in a form that allows the desired function to complete its action. If you want to find different summary statistics for different columns, the `.agg()` function can help. 

Because we added text columns to our `apple` dataset, calling `apple.apply(np.mean)` or something similar will fail. So, let's subset our `apple` dataset and then try using `.apply()` on the entire `DataFrame`.

In [ ]:
# Import numpy with alias np
import numpy as np

In [ ]:
# Change the output format for floats to be 2 decimals
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
# Get rid of nonnumerical columns
only_num_apple = apple.select_dtypes(include=np.number)
only_num_apple.info()

In [ ]:
# Find the average for all columns with .apply()
only_num_apple.apply(np.mean)

In [ ]:
# Now use .agg to find different summary statistics
# for different columns. Because picking out columns
# you can use the apple dataset
apple.agg({'open':np.mean, 'high':np.max, 'low':np.min,
          'close':np.mean, 'volume':np.sum})

In [ ]:
# If you want to use more than one statistic
# for a column, pass them as a list
apple.agg({'open':'mean',
          'high':['min','max'],
          'low':['min','max'],
          'close':'mean'})

<hr style="border:1px solid gray">

## Aggregating by Group

There are many instances where you want to calculate aggregations per group. To do so, we can use the `.groupby()` function, passing in the column(s) we want to use to determine distinct groups. Let's add the trading volume groups to our `apple` dataset. We will use the `pd.cut()` function as we did earlier.

In [ ]:
# Add the binned groups to the data
apple = apple.assign(vol_group = pd.cut(apple.volume, bins=3,
                   labels=['low','medium','high']))

apple.info()         

In [ ]:
# Let's find the average for each  vol_group
apple.groupby('vol_group').mean()

In [ ]:
# We can also pick out one of the columns from the resulting
# groupby and do specific aggregations on that column
apple.groupby('vol_group')['close'].agg(['min','max','mean'])

In [ ]:
# If you want to pull out different summary statistics
# for different columns, you can use .agg
app_agg = apple.groupby('vol_group').agg({
    'open':'mean', 'high':['min','max'],
    'low':['min','max'], 'close':'mean'})

app_agg

In [ ]:
# See the columns of app_agg
app_agg.columns

In [ ]:
# If you want to "flatten" the columns, one approach
# is to use list comprehension as shown here
app_agg.columns = ['_'.join(col_agg) for col_agg in app_agg.columns]
app_agg

You can also use a `pd.Grouper`, which allows you to specify a groupby instruction for an object. This is especially useful when you have an index that is a date. Our `apple` dataset has a date for an index. Suppose we wanted to find the average for the open, high, low, and close stock price for each month. This is easily accomplished with a `Grouper` by specifying the frequency. 

In [ ]:
# Find the aveage open, high, low, and close price for each month
apple.groupby(pd.Grouper(freq='M')).mean()[['open','high','low','close']]

<hr style="border:1px solid gray">

## Pivot Tables and Crosstabs

You are probably familiar with the powerful concept of pivot tables from Excel. We can quickly generate pivot tables or cross tabulations in a commonly used format with `pandas`. You need to specify what to group on and, optionally, which subset of columns we want to aggregate and/or how to aggregate. The default aggregation is **average**.

In [ ]:
# Call pivot_table with a grouping vol_group
apple.pivot_table(columns='vol_group')

If you want the transpose of this pivot table, you can specify `index='vol_group'` instead. Let's see if it works.

In [ ]:
# Put the vol_group categories on the rows
apple.pivot_table(index='vol_group')

The function `pd.crosstab()` can be used to create a frequency table. For example, suppose we want to know how many low-, medium-, and high-volume trading days occurred in each month of our dataset. You use `index` to specify the rows and `columns` to specify the columns. By default, the values in the cells will be the count.

In [ ]:
# Create a frequency table
# Rows will be the vol_group
# Columns will be the month number
pd.crosstab(index=apple.vol_group, columns=apple.index.month,
           colnames=['month'])

In [ ]:
# If you want totals for the rows and columns
# you can use the argument margins=True
pd.crosstab(index=apple.vol_group, columns=apple.index.month,
           colnames=['month'], margins=True)

In [ ]:
# If you want percentages by the month, you can use
# the argument normalize='columns'
pd.crosstab(index=apple.vol_group, columns=apple.index.month,
           colnames=['month'], normalize='columns')

In [ ]:
# If you want the overall percentage for each group over 
# the year, you can use normalize='columns' and margins=True
pd.crosstab(index=apple.vol_group, columns=apple.index.month,
           colnames=['month'], normalize='columns', margins=True)

<hr style="border:1px solid gray">

### Additional Resources

The following links point you to additional resources that you might find helpful in learning this material.

1. [The official API reference for `pandas.DataFrame.query`][1].
2. [The official API reference for `pandas.Series.pct_change`][2].
3. [The official API reference for `pandas.cut`][3].
4. [The official API reference for `pandas.qcut`][4].
5. [The official API reference for `pandas.Grouper`][5].

-----

[1]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html
[2]: https://pandas.pydata.org/docs/reference/api/pandas.Series.pct_change.html
[3]: https://pandas.pydata.org/docs/reference/api/pandas.cut.html
[4]: https://pandas.pydata.org/docs/reference/api/pandas.qcut.html
[5]: https://pandas.pydata.org/docs/reference/api/pandas.Grouper.html

-----

**&copy; 2022 - Present: Matthew D. Dean, Ph.D.   
Clinical Associate Professor of Business Analytics at William \& Mary.**